# Korzystanie z modeli OpenAI w Pythonie

## Korzystanie z API OpenAI

Standardową metodą korzystania z modeli OpenAI w Pythonie jest pakiet **openai**. Biblioteka ta umożliwia bezpośrednią komunikację z API OpenAI, dając dostęp do wszystkich modeli i funkcjonalności oferowanych przez platformę. Pakiet jest oficjalnie wspierany przez OpenAI i regularnie aktualizowany o nowe funkcje.

### Dwa główne API OpenAI

OpenAI udostępnia dwa różne API do interakcji z modelami:

1. **Chat Completions API** - standardowe, najpopularniejsze API (standard branżowy)
2. **Response API** - nowoczesne API z zaawansowanymi funkcjami agentycznymi

**Porównanie API:**

| Aspekt | Chat Completions API | Response API |
|--------|---------------------|--------------|
| **Endpoint** | `client.chat.completions.create()` | `client.responses.create()` |
| **Parametr wiadomości** | `messages=` | `input=` (string lub lista) |
| **Limit tokenów** | `max_completion_tokens=` | `max_output_tokens=` |
| **Dostęp do odpowiedzi** | `response.choices[0].message.content` | `response.output_text` |
| **Struktura outputu** | `choices` (lista Messages) | `output` (lista Items - różne typy) |
| **Statystyki tokenów** | `prompt_tokens`, `completion_tokens` | `input_tokens`, `output_tokens` |
| **Modalności outputu** | Text, audio (z gpt-realtime) | Text, images, (+ audio "soon") |
| **Dostęp do reasoning** | ❌ Brak | ✅ `reasoning.summary` |
| **Wbudowane narzędzia** | ❌ Tylko custom functions | ✅ web_search, file_search, code_interpreter, computer_use, MCP |
| **Zarządzanie stanem** | ❌ Ręczne | ✅ `previous_response_id`, Conversations API |
| **Parametry specjalne** | - | `store=True`, `background=True`, `instructions=` |
| **Wydajność cache** | Standardowa | 40-80% lepsze wykorzystanie |
| **Użycie** | Standardowe aplikacje czatowe, audio | Zaawansowane agenty, reasoning, automatyzacja |

**Kiedy używać którego API:**
- **Chat Completions API** - standardowe aplikacje czatowe, gdy potrzebujesz audio output, kompatybilność ze standardem branżowym
- **Response API** - zaawansowane agenty z narzędziami, reasoning, lepsze zarządzanie stanem, niższe koszty dzięki cache

Poniżej znajdują się przykłady użycia obu API oraz biblioteki LangChain.

In [ ]:
### 1. Chat Completions API

from openai import OpenAI
import os

# Inicjalizacja klienta
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

# Wysłanie zapytania
response = client.chat.completions.create(
    model="gpt-5.2",
    messages=[
        {"role": "system", "content": "Jesteś pomocnym poetą."},
        {"role": "user", "content": "Napisz wiersz o Gen AI"}
    ],
    temperature=0.7,
    max_completion_tokens=1000
)

# Wyświetlenie odpowiedzi
print(response.choices[0].message.content)

In [ ]:
### 2. Response API

from openai import OpenAI
import os

# Inicjalizacja klienta (ta sama co wcześniej)
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

# Wysłanie zapytania
response = client.responses.create(
    model="gpt-5.2",
    input=[  # RÓŻNICA: input zamiast messages
        {"role": "system", "content": "Jesteś pomocnym poetą."},
        {"role": "user", "content": "Napisz wiersz o Gen AI"}
    ],
    temperature=0.7,
    max_output_tokens=1000  # RÓŻNICA: max_output_tokens zamiast max_completion_tokens
)

# Wyświetlenie odpowiedzi
print(response.output_text)  # RÓŻNICA: output_text zamiast choices[0].message.content

## 2. LangChain

LangChain to popularna biblioteka tworząca abstrakcję nad różnymi dostawcami API modeli językowych. W teorii upraszcza budowanie aplikacji AI i łączenie ich w łańcuchy przetwarzania.

W praktyce jednak LangChain jest rzadko stosowany w zastosowaniach produkcyjnych. Wiele zespołów preferuje bezpośrednie API dostawców modeli ze względu na większą kontrolę, przewidywalność działania oraz niższe koszty utrzymania. LangChain, pomimo swojej elastyczności, często wprowadza dodatkową warstwę złożoności.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os

# Inicjalizacja klienta
api_key = os.getenv('OPENAI_API_KEY')
chat_model = ChatOpenAI(
    openai_api_key=api_key,
    model_name="gpt-5.2",
    temperature=0.7,
    max_tokens=1000
)

# Utworzenie promptu i łańcucha
prompt = ChatPromptTemplate.from_messages([
    ("system", "Jesteś pomocnym poetą."),
    ("human", "Napisz wiersz o Gen AI")
])

chain = prompt | chat_model

# Wywołanie łańcucha i wyświetlenie odpowiedzi
result = chain.invoke({})
print(result.content)